# 2) Feature selection

Reads `artifacts/windows.npz` (produced by the exploration notebook). 


Steps of feature-engineering: 
 
1. extract a broad set of candidate features per window
2. **split grouped by recording** (never by window)
3. filters: variance threshold, correlation, Fisher scatter score, mutual information
4. wrapper: greedy forward selection on cross-validated error rate
5. extraction: PCA
6. summary


In [23]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROC_DIR = Path("./../artifacts")
FIG_DIR = Path("./../figures")

z = np.load(PROC_DIR / "windows.npz")
X_raw, CHANNELS, FS = z["X"], list(z["channels"]), int(z["fs"])
meta = pd.read_csv(PROC_DIR / "windows_meta.csv")
y = (meta["label"] == "bumpy").to_numpy().astype(int)      # 1 = bumpy, 0 = smooth

print(f"X_raw {X_raw.shape}, channels {CHANNELS}, {FS} Hz, {y.sum()} bumpy / {(1 - y).sum()} smooth windows")

CLASSES = ["bumpy", "smooth"]
CLASS_COLOR = {"bumpy": "#eb6834", "smooth": "#2a78d6"}
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a"]
plt.rcParams.update(
    {"axes.grid": True, "grid.color": "#e1e0d9",
     "axes.spines.top": False, "axes.spines.right": False,
     "lines.linewidth": 1.0, "figure.dpi": 100})

X_raw (570, 400, 5), channels [np.str_('acc_vertical'), np.str_('acc_vertical_filtered'), np.str_('acc_horizontal'), np.str_('gyro_mag'), np.str_('tacc_mag')], 100 Hz, 285 bumpy / 285 smooth windows


## 2.1 Candidate features

Per window and channel: time-domain statistics; for the vertical channel and the gyroscope also spectral features
(band powers, band fractions, centroid, dominant frequency, spectral entropy). Generate generously here - the selection
steps below decide what survives.

In [24]:
BANDS = [(0.5, 3), (3, 8), (8, 15), (15, 30), (30, 50)]

# 8 temporal statistical features 
def time_feats(x, name):
    x = x - x.mean();       #Zero-Mean Normalization
    s = x.std() + 1e-12 
    
    return {f"{name}_std": x.std(), f"{name}_iqr": np.subtract(*np.percentile(x, [75, 25])), f"{name}_p2p": np.ptp(x),
            f"{name}_p95abs": np.percentile(np.abs(x), 95), f"{name}_madiff": np.abs(np.diff(x)).mean(),
            f"{name}_kurt": ((x / s) ** 4).mean(), f"{name}_skew": ((x / s) ** 3).mean(),
            f"{name}_zcr": (np.diff(np.sign(x)) != 0).mean()}

def spec_feats(x, name):
    w = np.hanning(len(x))
    P = np.abs(np.fft.rfft((x - x.mean()) * w)) ** 2
    f = np.fft.rfftfreq(len(x), 1 / FS)
    
    tot = P[f >= 0.5].sum() + 1e-12
    out = {}
    
    for lo, hi in BANDS:
        out[f"{name}_bp{lo}-{hi}"] = P[(f >= lo) & (f < hi)].sum()           # absolute band power
        out[f"{name}_bf{lo}-{hi}"] = out[f"{name}_bp{lo}-{hi}"] / tot        # fraction of total
        
    p = P[f >= 0.5] / tot
    out[f"{name}_centroid"] = (f[f >= 0.5] * p).sum()
    out[f"{name}_domfreq"] = f[f >= 0.5][p.argmax()]
    out[f"{name}_spec_entropy"] = -(p[p > 0] * np.log(p[p > 0])).sum()
    return out

def extract_features(w):
    """w: one window (samples, channels) in CHANNELS order -> dict of features."""
    ch = {c: w[:, i] for i, c in enumerate(CHANNELS)}
    return (time_feats(ch["acc_vertical"], "vert") 
            | time_feats(ch["acc_vertical_filtered"], "vertbp") | time_feats(ch["acc_horizontal"], "horiz")
            | time_feats(ch["gyro_mag"], "gyro") | time_feats(ch["tacc_mag"], "tacc")
            | spec_feats(ch["acc_vertical"], "vert") | spec_feats(ch["gyro_mag"], "gyro"))

feat = pd.DataFrame([extract_features(w) for w in X_raw])
FEATS = list(feat.columns)
X = feat.to_numpy(float)

print(f"{len(FEATS)} candidate features for {len(feat)} windows")
feat.describe().T.round(3).head(12)

66 candidate features for 570 windows


,count,mean,std,min,25%,50%,75%,max
vert_std,570.0,2.361,1.072,0.503,1.575,2.361,3.118,5.218
vert_iqr,570.0,2.929,1.382,0.612,1.844,2.903,4.021,6.450
vert_p2p,570.0,14.688,7.846,2.937,9.325,13.832,18.598,45.429
vert_p95abs,570.0,4.639,2.107,0.939,3.239,4.607,6.125,10.543
vert_madiff,570.0,0.689,0.336,0.190,0.404,0.621,0.949,1.915
vert_kurt,570.0,4.089,3.422,2.018,2.867,3.340,4.241,54.986
vert_skew,570.0,0.078,0.594,-1.572,-0.276,0.041,0.424,3.777
vert_zcr,570.0,0.124,0.046,0.038,0.088,0.118,0.158,0.268
vertbp_std,570.0,1.774,0.950,0.417,0.939,1.614,2.451,4.410
vertbp_iqr,570.0,2.132,1.128,0.516,1.118,1.838,3.094,5.369
